In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import yaml
from sklearn.metrics import accuracy_score, recall_score, precision_score
from src.dynamic_pipeline_builder import DynamicPipelineBuilder, validate_config

# --- 1. Завантаження даних та обчислення Non-ML Benchmark ---
# Примітка: Завантажуємо cs-training.csv з Give Me Some Credit
df = pd.read_csv("cs-training.csv", index_col=0)

In [2]:
# Цільова змінна: SeriousDlqin2yrs (1 - дефолт/bad, 0 - без дефолту/good)
y_true = df["SeriousDlqin2yrs"].values

# Формулювання детермінованого евристичного правила без ML:
# Якщо людина має низький місячний дохід (< 1000) АБО високу прострочку -> вважаємо її потенційним дефолтником (1)
heuristic_rule = (df["MonthlyIncome"] < 1000) | (df["NumberOfTimes90DaysLate"] > 0)
y_heuristic = heuristic_rule.astype(int).values

acc = accuracy_score(y_true, y_heuristic)
rec = recall_score(y_true, y_heuristic, pos_label=1)
prec = precision_score(y_true, y_heuristic, pos_label=1)

print("=== NON-ML HEURISTIC BENCHMARK RESULTS ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"Precision: {prec:.4f}")

=== NON-ML HEURISTIC BENCHMARK RESULTS ===
Accuracy:  0.8978
Recall:    0.3599
Precision: 0.2881


In [3]:
# --- 2. Демонстрація спрацювання Quality Gate (validate_config) ---
print("\n=== TESTING QUALITY GATE (validate_config) ===")

# Створюємо навмисно зіпсовану конфігурацію, де 'age' (захищена ознака) внесена у features
corrupted_config = {
    "model_family": "additive",
    "features": {
        "numeric_continuous": ["age", "MonthlyIncome", "DebtRatio"] # LEAK! 'age' тут!
    },
    "protected_attributes": ["age"]
}

try:
    print("Спроба валідації зіпсованої конфігурації...")
    validate_config(corrupted_config)
except ValueError as e:
    print("\n[УСПІХ] Quality Gate спрацював коректно та підняв ValueError:")
    print(e)


=== TESTING QUALITY GATE (validate_config) ===
Спроба валідації зіпсованої конфігурації...

[УСПІХ] Quality Gate спрацював коректно та підняв ValueError:
[QUALITY GATE ERROR] Захищені ознаки потрапили у features: {'age'}. Необхідно вилучити їх з секції features у YAML конфігурації!


In [6]:
# Обчислення кореляції між age та іншими числовими ознаками
correlations = df.corr()["age"].sort_values()
correlations

NumberOfDependents                     -0.213303
SeriousDlqin2yrs                       -0.115386
NumberOfTime30-59DaysPastDueNotWorse   -0.062995
NumberOfTimes90DaysLate                -0.061005
NumberOfTime60-89DaysPastDueNotWorse   -0.057159
RevolvingUtilizationOfUnsecuredLines   -0.005898
DebtRatio                               0.024188
NumberRealEstateLoansOrLines            0.033150
MonthlyIncome                           0.037717
NumberOfOpenCreditLinesAndLoans         0.147705
age                                     1.000000
Name: age, dtype: float64

In [4]:
X = df.drop(columns=["SeriousDlqin2yrs"])
X_demo = X.iloc[:120000]
ct = DynamicPipelineBuilder("../config/config_lr.yaml").build().fit(X_demo)
imp = ct.named_transformers_["numeric_continuous"].named_steps["adaptive_imputer"]
for col, rec in imp.audit_trail_.items():
    print(f"{col}: {rec['missing_share']:.4f} -> {rec['applied_rule']}")

RevolvingUtilizationOfUnsecuredLines: 0.0000 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
DebtRatio: 0.0000 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
MonthlyIncome: 0.1981 -> {'max_share': 0.4, 'strategy': 'median', 'add_indicator': True}
NumberOfOpenCreditLinesAndLoans: 0.0000 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
NumberRealEstateLoansOrLines: 0.0000 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
NumberOfDependents: 0.0262 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
NumberOfTime30-59DaysPastDueNotWorse: 0.0000 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
NumberOfTime60-89DaysPastDueNotWorse: 0.0000 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
NumberOfTimes90DaysLate: 0.0000 -> {'max_share': 0.1, 'strategy': 'median', 'add_indicator': False}
